# Task 4 — Baseline: DINOv2 ViT-S/14 (frozen) + Logistic Regression
ดึง feature จาก DINOv2 แบบไม่ fine-tune (CLS + mean patch = 768 มิติ) → Logistic Regression (class_weight=balanced)
วัดผลด้วย 5-fold Stratified CV, **macro F1** · ไม่ submit — เขียนไฟล์ลง `/kaggle/working/` เท่านั้น

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd, torch, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")
SEED, IMG, BS, MODEL = 42, 224, 256, "vit_small_patch14_dinov2.lvd142m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| timm", timm.__version__)

def walk_find(name, base="/kaggle/input"):
    hits = []
    for d, _, files in os.walk(base, followlinks=True):
        if name in files: hits.append(os.path.join(d, name))
    return sorted(hits)
for d, dirs, files in os.walk("/kaggle/input", followlinks=True):      # แสดงโครงสร้าง (ไม่ลงไปในโฟลเดอร์รูป)
    if d.count(os.sep) <= 5 and not d.endswith(("/train", "/test")):
        print(d, "|", len(files), "files", [f for f in files if f.endswith(".csv")][:5])
cands = [h for h in walk_find("train.csv") if os.path.isdir(os.path.join(os.path.dirname(h), "train"))]
print("train.csv candidates:", cands)
root = os.path.dirname(cands[0])
SAMPLE = walk_find("sample_submission.csv")
print("data:", root, "| sample:", SAMPLE[:1])
train = pd.read_csv(f"{root}/train.csv"); test = pd.read_csv(f"{root}/test.csv")
CLASSES = sorted(train["class"].unique())          # ['7-11k','fiveN','freefried','pube','roblock'] = datacard 0..4
cid = {c: i for i, c in enumerate(CLASSES)}
y = train["class"].map(cid).values
print(train.shape, test.shape, CLASSES); print(train["class"].value_counts().to_dict())

In [ ]:
class Imgs(Dataset):
    def __init__(self, folder, ids, tf): self.f, self.ids, self.tf = folder, list(ids), tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.tf(Image.open(f"{self.f}/{self.ids[i]}.jpg").convert("RGB"))

model = timm.create_model(MODEL, pretrained=True, num_classes=0, img_size=IMG).to(DEVICE).eval()
cfg = timm.data.resolve_model_data_config(model)
import torchvision.transforms as T
tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])])

@torch.no_grad()
def extract(folder, ids):
    dl = DataLoader(Imgs(folder, ids, tf), batch_size=BS, num_workers=4, pin_memory=True)
    out = []
    for x in dl:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            t = model.forward_features(x.to(DEVICE, non_blocking=True))   # [B, 1+N, 384]
        out.append(torch.cat([t[:, 0], t[:, 1:].mean(1)], 1).float().cpu())
    return torch.cat(out).numpy()

t0 = time.time()
Xtr = extract(f"{root}/train", train["id"]); print("train feats", Xtr.shape, f"{time.time()-t0:.0f}s")
Xte = extract(f"{root}/test", test["id"]);   print("test feats", Xte.shape, f"{time.time()-t0:.0f}s")
np.save("/kaggle/working/dinov2s_train.npy", Xtr); np.save("/kaggle/working/dinov2s_test.npy", Xte)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def make_clf(C): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced", n_jobs=-1))
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
folds = list(skf.split(Xtr, y))

# เลือก C เร็ว ๆ บน fold แรก
tr, va = folds[0]
sweep = {}
for C in (0.05, 0.2, 1.0):
    p = make_clf(C).fit(Xtr[tr], y[tr]).predict(Xtr[va])
    sweep[C] = f1_score(y[va], p, average="macro"); print(f"C={C}: fold0 macro F1 = {sweep[C]:.4f}")
BEST_C = max(sweep, key=sweep.get); print("BEST_C =", BEST_C)

In [ ]:
oof = np.zeros((len(y), len(CLASSES)))
for k, (tr, va) in enumerate(folds):
    clf = make_clf(BEST_C).fit(Xtr[tr], y[tr]); oof[va] = clf.predict_proba(Xtr[va])
    print(f"fold {k}: macro F1 {f1_score(y[va], oof[va].argmax(1), average='macro'):.4f}")
pred = oof.argmax(1)
print(f"\nOOF macro F1 = {f1_score(y, pred, average='macro'):.4f}\n")
print(classification_report(y, pred, target_names=CLASSES, digits=4))
print(pd.DataFrame(confusion_matrix(y, pred), index=CLASSES, columns=CLASSES))
np.save("/kaggle/working/dinov2s_oof_proba.npy", oof)

In [ ]:
clf = make_clf(BEST_C).fit(Xtr, y)
proba = clf.predict_proba(Xte); pt = proba.argmax(1)
os.makedirs("/kaggle/working/preds", exist_ok=True); os.makedirs("/kaggle/working/probes", exist_ok=True)
np.save("/kaggle/working/preds/task4_dinov2s_test_proba.npy", proba)
out = pd.DataFrame({"id_img": test["id"], "pred_num": pt, "pred_str": [CLASSES[i] for i in pt]})
print(out["pred_str"].value_counts(normalize=True).round(3).to_dict())

if SAMPLE:
    ids = pd.read_csv(SAMPLE[0], sep=None, engine="python")["id"]
    assert len(ids) == len(test), "จำนวนแถว test ≠ sample_submission"
    out.insert(0, "id", ids.values)          # จับคู่ตามลำดับแถว (เหมือน task อื่น)
    for fmt in ("num", "str"):
        pd.DataFrame({"id": ids, "pred": out[f"pred_{fmt}"]}).to_csv(f"/kaggle/working/preds/task4_dinov2s_lr_{fmt}.csv", index=False)
        # probe: mask task 1,2,3,5 (คะแนน 0 / task3 = 0.5) + task4 จริง → ใช้คู่กับ P0_all_mask.csv
        pd.DataFrame({"id": ids, "task1": 2, "task2": 9, "task3": 0, "task4": out[f"pred_{fmt}"], "task5": 2}) \
          .to_csv(f"/kaggle/working/probes/P4_task4_only_mask_dinov2s_{fmt}.csv", index=False)
    print("เขียน preds/ และ probes/ (ทั้งแบบตัวเลข 0–4 และแบบข้อความ) — ไม่ได้ submit")
out.to_csv("/kaggle/working/preds/task4_dinov2s_lr_full.csv", index=False)